In [ ]:
import os
import random
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid")

In [ ]:
project_dir = Path.cwd()
DATA_PATH = project_dir / "data" / "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst rows:")
display(df.head())

# Clean column names and keep only usable features.
df.columns = df.columns.str.strip()
metadata_cols = ["Flow ID", "Source IP", "Destination IP", "Timestamp"]
metadata_cols = [col for col in metadata_cols if col in df.columns]
df = df.drop(columns=metadata_cols)

df = df.replace([np.inf, -np.inf], np.nan)

for col in df.columns:
    if col == "Label":
        continue
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Replace impossible values and missing data with safe medians.
for col in df.columns:
    if col == "Label":
        continue
    df[col] = df[col].replace([np.inf, -np.inf], np.nan)
    median = df[col].median()
    if pd.isna(median):
        df[col] = df[col].fillna(0)
    else:
        df[col] = df[col].fillna(median)

# Label encoding.
df["Label"] = df["Label"].astype(str).str.strip()
df["Label"] = df["Label"].map({"BENIGN": 0, "DDoS": 1})
print("\nLabel distribution:")
print(df["Label"].value_counts().sort_index())

X = df.drop(columns=["Label"])
y = df["Label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

models = {
    "Logistic Regression": Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                    random_state=SEED,
                    class_weight="balanced",
                ),
            ),
        ]
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=SEED,
        class_weight="balanced",
        n_jobs=-1,
    ),
    "MLP": Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "model",
                MLPClassifier(
                    hidden_layer_sizes=(64, 32),
                    max_iter=500,
                    random_state=SEED,
                ),
            ),
        ]
    ),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    result = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
    }

    if hasattr(model, "predict_proba"):
        try:
            result["roc_auc"] = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
        except Exception:
            pass

    results[name] = result

    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred, target_names=["BENIGN", "DDoS"]))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

summary = pd.DataFrame(results).T.sort_values("f1", ascending=False)
print("\nSummary of model performance:")
print(summary)

rf_model = models["Random Forest"]
feature_importance = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 10 features:")
print(feature_importance.head(10))

fig_dir = project_dir / "figures"
fig_dir.mkdir(exist_ok=True)

feature_importance.head(15).sort_values().plot(kind="barh", figsize=(10, 6), color="steelblue")
plt.title("Top 15 features for DDoS detection")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(fig_dir / "top_features.png", dpi=150)
plt.close()

results_dir = project_dir / "results"
results_dir.mkdir(exist_ok=True)
summary.to_csv(results_dir / "model_metrics.csv")
print("\nSaved metrics to:", results_dir / "model_metrics.csv")
print("Saved feature plot to:", fig_dir / "top_features.png")